In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

BASE_PATH = Path(
    "/content/drive/MyDrive/ECs Master Folder/Research/"
    "Kidney Thermal Property MFNN/Data/raw"
)

# nested dictionaries
# outer dictionary (text as key) -> dictionaries as the value
# that dictionary value has key-value pairs
EXPERIMENTS = {
    "porcine_conductivity": {
        "file": "porcine_raw.csv",
        "target": "conductivity_mean_W_mK",
        "uncertainty": "conductivity_EU95_W_mK",
    },
    "porcine_diffusivity": {
        "file": "porcine_raw.csv",
        "target": "diffusivity_mean_mm2_s",
        "uncertainty": "diffusivity_EU95_mm2_s",
    },
    "bovine_conductivity": {
        "file": "bovine_raw.csv",
        "target": "conductivity_mean_W_mK",
        "uncertainty": "conductivity_EU95_W_mK"
    },
    "bovine_diffusivity": {
        "file": "bovine_raw.csv",
        "target": "diffusivity_mean_mm2_s",
        "uncertainty": "diffusivity_EU95_mm2_s",
    },
}

TEMP_COL = "temperature_C"
HF_SIZES = [3, 5, 7, 9]
SEEDS = range(20)

In [22]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# returns a pandas dataframe w 3 columns of the hi-fi data
def load_experiment(config): # config = one inner dictionary
  df = pd.read_csv(BASE_PATH / config["file"]) # opens csv file

  required_columns = [
      TEMP_COL,
      config["target"],
      config["uncertainty"]
  ]

  missing_columns = []

  # adds column
  for column in required_columns:
    if column not in df.columns:
        missing_columns.append(column) # stores a variable w/ missing column names

  if missing_columns: # non-empty list -> True, empty list -> False
    raise ValueError(
        f"Missing columns: {missing_columns}\n"
        f"Available columns: {list(df.columns)}"
    )

  return (
      df[required_columns]
      .dropna() # deletes rows w/ missing values
      .sort_values(TEMP_COL) # low -> high
      .reset_index(drop=True) # resets row indices
  )

In [24]:
for experiment_name, config in EXPERIMENTS.items():
  data = load_experiment(config) # config = value

  print(experiment_name) # key
  print("Number of HF points:", len(data))
  display(data)

porcine_conductivity
Number of HF points: 11


,temperature_C,conductivity_mean_W_mK,conductivity_EU95_W_mK
0,23.9,0.549,0.045
1,35.4,0.559,0.050
2,41.5,0.573,0.050
3,46.2,0.573,0.030
4,56.7,0.571,0.033
5,60.0,0.560,0.022
6,70.1,0.536,0.017
7,76.4,0.528,0.026
8,82.3,0.527,0.080
9,86.6,0.596,0.058


porcine_diffusivity
Number of HF points: 11


,temperature_C,diffusivity_mean_mm2_s,diffusivity_EU95_mm2_s
0,23.9,0.155,0.019
1,35.4,0.157,0.011
2,41.5,0.161,0.009
3,46.2,0.163,0.011
4,56.7,0.161,0.007
5,60.0,0.160,0.008
6,70.1,0.165,0.011
7,76.4,0.171,0.014
8,82.3,0.178,0.016
9,86.6,0.194,0.033


bovine_conductivity
Number of HF points: 10


,temperature_C,conductivity_mean_W_mK,conductivity_EU95_W_mK
0,23.2,0.528,0.066
1,30.2,0.548,0.030
2,36.9,0.551,0.091
3,40.9,0.528,0.180
4,48.8,0.541,0.129
5,56.4,0.562,0.167
6,69.5,0.595,0.183
7,75.8,0.590,0.164
8,81.1,0.550,0.191
9,90.0,0.703,0.226


bovine_diffusivity
Number of HF points: 10


,temperature_C,diffusivity_mean_mm2_s,diffusivity_EU95_mm2_s
0,23.2,0.151,0.020
1,30.2,0.153,0.030
2,36.9,0.157,0.035
3,40.9,0.157,0.032
4,48.8,0.160,0.043
5,56.4,0.155,0.015
6,69.5,0.162,0.029
7,75.8,0.169,0.031
8,81.1,0.160,0.009
9,90.0,0.183,0.006


In [25]:
def split_hf_data(data, n_hf, seed):
  rng = np.random.default_rng(seed)

  all_indices = np.arange(len(data)) # creates all row positions in a list

  # same row can't be selected twice
  # picking n_hf indices
  train_indices = rng.choice(
      all_indices,
      size=n_hf,
      replace=False
  )

  # selects all indices that were not put into train_indices
  test_indices = np.setdiff1d(
      all_indices,
      train_indices
  )

  # selects all rows in pandas df w/ those row #s, renumbers rows starting from 0
  train_data = data.iloc[train_indices].reset_index(drop=True)
  test_data = data.iloc[test_indices].reset_index(drop=True)

  return train_data, test_data

In [26]:
# test / proof of concept
config = EXPERIMENTS["porcine_conductivity"]
data = load_experiment(config)

train_data, test_data = split_hf_data(
    data,
    n_hf=3,
    seed=0,
)

print("Training data:")
display(train_data)

print("Testing data:")
display(test_data)

print("Training points:", len(train_data))
print("Testing points:", len(test_data))
print("Combined points:", len(train_data) + len(test_data))
print("Original points:", len(data))

Training data:


,temperature_C,conductivity_mean_W_mK,conductivity_EU95_W_mK
0,70.1,0.536,0.017
1,60.0,0.560,0.022
2,76.4,0.528,0.026


Testing data:


,temperature_C,conductivity_mean_W_mK,conductivity_EU95_W_mK
0,23.9,0.549,0.045
1,35.4,0.559,0.050
2,41.5,0.573,0.050
3,46.2,0.573,0.030
4,56.7,0.571,0.033
5,82.3,0.527,0.080
6,86.6,0.596,0.058
7,92.8,0.648,0.061


Training points: 3
Testing points: 8
Combined points: 11
Original points: 11


In [27]:
from sklearn.linear_model import LinearRegression

def fit_low_fidelity(train_data, target_col): # target_col is the name of the output column we want to predict
  model = LinearRegression()

  x_train = train_data[[TEMP_COL]].to_numpy()
  y_train = train_data[target_col].to_numpy()

  model.fit(x_train, y_train)

  return model

In [28]:
lf_model = fit_low_fidelity(
    train_data,
    config["target"],
)

print("Slope:", lf_model.coef_[0]) # one slope because it's simple (minimal input features)
print("Intercept:", lf_model.intercept_)

Slope: -0.0019909414113865494
Intercept: 0.6783764671504409


In [29]:
train_lf_predictions = lf_model.predict(
    train_data[[TEMP_COL]].to_numpy() # uses model to predict training data -> temp, will be used in MFNN train (temp, LF pred, correct HF)
)

test_lf_predictions = lf_model.predict(
    test_data[[TEMP_COL]].to_numpy() # test data
)

print("Training LF predictions:")
print(train_lf_predictions)

print("Testing LF predictions")
print(test_lf_predictions)

Training LF predictions:
[0.53881147 0.55891998 0.52626854]
Testing LF predictions
[0.63079297 0.60789714 0.5957524  0.58639497 0.56549009 0.51452199
 0.50596094 0.4936171 ]


In [30]:
test_comparison = test_data[ # grabbing these columns from test_data
    [TEMP_COL, config["target"]] # temperature_C, conductivity_mean_W_mK
].copy() # making a copy so we don't edit the actual test_data

test_comparison["lf_prediction"] = test_lf_predictions
test_comparison["abs_error"] = abs(test_data[config["target"]]-test_lf_predictions)

display(test_comparison)

,temperature_C,conductivity_mean_W_mK,lf_prediction,abs_error
0,23.9,0.549,0.630793,0.081793
1,35.4,0.559,0.607897,0.048897
2,41.5,0.573,0.595752,0.022752
3,46.2,0.573,0.586395,0.013395
4,56.7,0.571,0.565490,0.005510
5,82.3,0.527,0.514522,0.012478
6,86.6,0.596,0.505961,0.090039
7,92.8,0.648,0.493617,0.154383


In [31]:
test_mae = test_comparison["abs_error"].mean()

print("Test MAE:", test_mae)

test_rmse = np.sqrt(
    np.mean(
        (test_data[config["target"]] - test_lf_predictions)**2
    )
)

print("Test RMSE:", test_rmse)

Test MAE: 0.053655919738956816
Test RMSE: 0.07237514029594305


In [32]:
def calculate_metrics(y_true, y_pred):
  errors = y_true - y_pred

  mae = np.mean(np.abs(errors))
  rmse = np.sqrt(np.mean(errors**2))

  return mae, rmse

In [33]:
lf_mae, lf_rmse = calculate_metrics(
    test_data[config["target"]].to_numpy(),
    test_lf_predictions
)

print("LF MAE:", lf_mae)
print("LF RMSE:", lf_rmse)

LF MAE: 0.053655919738956816
LF RMSE: 0.07237514029594305


In [34]:
# train temps, train pred conductivity
x_mfnn_train = np.column_stack([
    train_data[TEMP_COL].to_numpy(),
    train_lf_predictions
])

# test temps, test pred conductivity
x_mfnn_test = np.column_stack([
    test_data[TEMP_COL].to_numpy(),
    test_lf_predictions
])

# target = conductivity_mean_W_mk (in this case specifically)
y_mfnn_train = train_data[config["target"]].to_numpy() # true training conductivity
y_mfnn_test = test_data[config["target"]].to_numpy() # true testing conductivity

print("MFNN training inputs:")
print(x_mfnn_train)

print("True training conductivities:")
print(y_mfnn_train)

print("Training input shape:", x_mfnn_train.shape)
print("Testing input shape:", x_mfnn_test.shape)

MFNN training inputs:
[[70.1         0.53881147]
 [60.          0.55891998]
 [76.4         0.52626854]]
True training conductivities:
[0.536 0.56  0.528]
Training input shape: (3, 2)
Testing input shape: (8, 2)


In [35]:
from sklearn.preprocessing import StandardScaler

temperature_scaler = StandardScaler()
target_scaler = StandardScaler()

temperature_train = train_data[[TEMP_COL]].to_numpy()
temperature_test = test_data[[TEMP_COL]].to_numpy()

### train data ###

# scaled train temps
temperature_train_scaled = temperature_scaler.fit_transform(
    temperature_train
)

# target values, single column
# StandardScaler requires a 2d array
y_train_column = y_mfnn_train.reshape(-1, 1)

y_train_scaled = target_scaler.fit_transform(
    y_train_column
)

# scaling lf predictions using same scaler
lf_train_scaled = target_scaler.transform(
    train_lf_predictions.reshape(-1, 1)
)

### test data ###

# scaled test temps
temperature_test_scaled = temperature_scaler.transform(
    temperature_test
)

# lo-fi test input scaled
lf_test_scaled = target_scaler.transform(
    test_lf_predictions.reshape(-1, 1)
)

# we don't scale target tests because we only compare to calculate mae and rmse during test
# we scale target train to calculate error / train

print("Scaled training temp:")
print(temperature_train_scaled)

print("\nScaled HF training targets:")
print(y_train_scaled)

print("\nScaled LF training predictions:")
print(lf_train_scaled)

print("\nShapes:")
print("Temperature train:", temperature_train_scaled.shape)
print("Temperature test:", temperature_test_scaled.shape)
print("HF target train:", y_train_scaled.shape)
print("LF train:", lf_train_scaled.shape)
print("LF test:", lf_test_scaled.shape)

Scaled training temp:
[[ 0.18751776]
 [-1.30768961]
 [ 1.12017185]]

Scaled HF training targets:
[[-0.39223227]
 [ 1.37281295]
 [-0.98058068]]

Scaled LF training predictions:
[[-0.18546647]
 [ 1.29338462]
 [-1.10791815]]

Shapes:
Temperature train: (3, 1)
Temperature test: (8, 1)
HF target train: (3, 1)
LF train: (3, 1)
LF test: (8, 1)


In [36]:
# converting to tensors so the nn can use them
# retains shape
temperature_train_tensor = torch.tensor(
    temperature_train_scaled,
    dtype=torch.float32
)

temperature_test_tensor = torch.tensor(
    temperature_test_scaled,
    dtype=torch.float32
)

lf_train_tensor = torch.tensor(
    lf_train_scaled,
    dtype=torch.float32
)

lf_test_tensor = torch.tensor(
    lf_test_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_scaled,
    dtype=torch.float32
)

print("Training temperature tensor:")
print(temperature_train_tensor)

print("\nTraining LF tensor:")
print(lf_train_tensor)

print("\nTraining target tensor:")
print(y_train_tensor)

print("\nTensor shapes:")
print("Temperature train:", temperature_train_tensor.shape)
print("Temperature test:", temperature_test_tensor.shape)
print("LF train:", lf_train_tensor.shape)
print("LF test:", lf_test_tensor.shape)
print("HF target train:", y_train_tensor.shape)

Training temperature tensor:
tensor([[ 0.1875],
        [-1.3077],
        [ 1.1202]])

Training LF tensor:
tensor([[-0.1855],
        [ 1.2934],
        [-1.1079]])

Training target tensor:
tensor([[-0.3922],
        [ 1.3728],
        [-0.9806]])

Tensor shapes:
Temperature train: torch.Size([3, 1])
Temperature test: torch.Size([8, 1])
LF train: torch.Size([3, 1])
LF test: torch.Size([8, 1])
HF target train: torch.Size([3, 1])


In [37]:
import torch.nn as nn

class ResidualMFNN(nn.Module):
  def __init__(self, hidden_units=4): # build the model
    super().__init__()

    # will learn how strong to weigh the lo-fi predictions
    self.rho = nn.Parameter(
        torch.tensor(1.0, dtype=torch.float32)
    )

    # learns correction from temperature, takes only temperature
    self.correction_network = nn.Sequential(
        nn.Linear(1, hidden_units),
        nn.Tanh(),
        nn.Linear(hidden_units, 1)
    )

    # makes sure we start with prediction of HF = LF
    nn.init.zeros_(self.correction_network[-1].weight)
    nn.init.zeros_(self.correction_network[-1].bias)

  def forward(self, temperature, lf_prediction): # use the model
    correction = self.correction_network(temperature)
    hf_prediction = (self.rho * lf_prediction + correction)

    return hf_prediction

model = ResidualMFNN(hidden_units=4)
loss_function = nn.MSELoss() # finds predictions, squares the error, averages

# optimizes parameters
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


In [38]:
def run_single_split(
    experiment_name,
    n_hf,
    seed,
    hidden_units=4,
    epochs=2000,
    learning_rate=0.01
):
  config = EXPERIMENTS[experiment_name]
  data = load_experiment(config)

  if n_hf >= len(data):
    raise ValueError(
        f"n_hf={n_hf} must be smaller than "
        f"the dataset size of {len(data)}"
    )

  train_data, test_data = split_hf_data(
      data=data,
      n_hf=n_hf,
      seed=seed
  )

  target_col = config["target"]

  # creates linear fit line on training HF
  # we know how many n to use to fit the line, we use train data
  # target_col just tells us the name of the column w/ value
  lf_model = fit_low_fidelity(
      train_data=train_data,
      target_col=target_col
  )

  # temps
  temperature_train = train_data[[TEMP_COL]].to_numpy()
  temperature_test = test_data[[TEMP_COL]].to_numpy()

  # target values
  y_train_real = train_data[target_col].to_numpy()
  y_test_real = test_data[target_col].to_numpy()

  lf_train_predictions = lf_model.predict(temperature_train)
  lf_test_predictions = lf_model.predict(temperature_test)

  lf_mae, lf_rmse = calculate_metrics(
      y_true=y_test_real,
      y_pred=lf_test_predictions
  )

  temperature_scaler = StandardScaler()
  target_scaler = StandardScaler()

  # we use transform of test data bc model learns based on train data's fit
  temperature_train_scaled = temperature_scaler.fit_transform(temperature_train)
  temperature_test_scaled = temperature_scaler.transform(temperature_test)

  y_train_scaled = target_scaler.fit_transform(
      y_train_real.reshape(-1,1)
  )

  # lf for train/test scaled on HF's scale
  lf_train_scaled = target_scaler.transform(
      lf_train_predictions.reshape(-1,1)
  )
  lf_test_scaled = target_scaler.transform(
      lf_test_predictions.reshape(-1,1)
  )

  temperature_train_tensor = torch.tensor(temperature_train_scaled, dtype=torch.float32)
  temperature_test_tensor = torch.tensor(temperature_test_scaled, dtype=torch.float32)
  lf_train_tensor = torch.tensor(lf_train_scaled, dtype=torch.float32)
  lf_test_tensor = torch.tensor(lf_test_scaled, dtype=torch.float32)
  y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

  # makes reproducible
  torch.manual_seed(seed)

  model = ResidualMFNN(hidden_units=hidden_units)
  loss_function = nn.MSELoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) # giving it all model parameters w gradients

  model.train()

  for epoch in range(epochs):
    optimizer.zero_grad()

    train_predictions = model(
        temperature_train_tensor,
        lf_train_tensor
    )

    loss = loss_function(
        train_predictions,
        y_train_tensor
    )

    loss.backward()
    optimizer.step()

  final_training_loss = loss.item() # loss on final epoch

  model.eval() # eval mode

  with torch.no_grad():
    test_predictions_scaled = model(
        temperature_test_tensor,
        lf_test_tensor
    )

  mfnn_test_predictions = target_scaler.inverse_transform(
      test_predictions_scaled.cpu().numpy() # target scaler expects a numpy
  ).ravel() # gets rid of the column dimension

  mfnn_mae, mfnn_rmse = calculate_metrics(
      y_true=y_test_real,
      y_pred=mfnn_test_predictions
  )

  return {
      "experiment name": experiment_name,
      "n_hf": n_hf,
      "seed": seed,
      "n_train": len(train_data),
      "n_test": len(test_data),
      "lf_mae": lf_mae,
      "lf_rmse": lf_rmse,
      "mfnn_mae": mfnn_mae,
      "mfnn_rmse": mfnn_rmse,
      "learned_rho": model.rho.item(),
      "final_training_loss": final_training_loss
  }

In [39]:
single_split_result = run_single_split(
    experiment_name="porcine_conductivity",
    n_hf=3,
    seed=0
)

pd.Series(single_split_result)

,0
experiment name,porcine_conductivity
n_hf,3
seed,0
n_train,3
n_test,8
lf_mae,0.053656
lf_rmse,0.072375
mfnn_mae,0.048867
mfnn_rmse,0.067139
learned_rho,0.863557


In [ ]:
# we are now running all 320 splits

import time

all_results = []

total_runs = len(EXPERIMENTS)*len(HF_SIZES)*len(SEEDS)

completed_runs = 0
start_time = time.time()

for experiment_name in EXPERIMENTS:
  for n_hf in HF_SIZES:
    for seed in SEEDS:
      result = run_single_split(
          experiment_name=experiment_name,
          n_hf=n_hf,
          seed=seed,
          hidden_units=4,
          epochs=2000,
          learning_rate=0.01
      )

      all_results.append(result)
      completed_runs+=1

      # update every 20 runs
      if completed_runs % 20 == 0:
        print(f"Completed {completed_runs}/{total_runs} runs")

elapsed_minutes = (time.time() - start_time)/60 # time it took to run all 320 splits

# each dictionary = row
# each dictionary key = column
results_df = pd.DataFrame(all_results)

# safety measures
assert len(results_df) == total_runs
assert not results_df.isnull().any().any() # (results_df.isnull().any().any()) returns false if no missing value, not makes it return true

RESULTS_PATH = BASE_PATH.parent / "results"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

raw_results_file = RESULTS_PATH / "mfnn_all_split_results.csv"

results_df.to_csv(
    raw_results_file,
    index=False # no df indexes
)

print("\nAll runs good")
print("Results shape:", results_df.shape)
print(f"Runtime: {elapsed_minutes:.2f} minutes")
print("Saved to:", raw_results_file)

results_df.head() # just so I can quickly inspect